<a href="https://colab.research.google.com/github/msainavtej/IIITHInternship2/blob/main/LoRAfinetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LegalDoc-PEFT: Legislative Summarization via LoRA Fine-Tuning

## 1. Problem Statement & Scope
* **Task:** Domain-specific summarization of dense legislative bills into structured, plain-English executive points.
* **Domain:** Legal & Legislative Documentation.
* **Challenge Addressed:** Raw legal texts contain heavy statutory jargon and nested clauses. General foundational models often produce overly generic or hallucinated summaries without specialized domain adaptation.

---

## 2. Dataset & Formatting
* **Benchmark:** `FiscalNote/billsum` (U.S. Congressional legislative texts).
* **Format:** Formatted using ChatML-style tokens (`<|im_start|>system / user / assistant<|im_end|>`) to maintain structural instruction alignment:
  * **System:** Role assignment (`"You are a legal expert..."`).
  * **User:** Truncated raw legislative bill text.
  * **Assistant:** Ground truth legal summary.

---

## 3. PEFT & Training Methodology

| Parameter | Configuration | Technical Rationale |
| :--- | :--- | :--- |
| **Base Architecture** | `Qwen/Qwen2.5-1.5B-Instruct` | High reasoning density at low parameter overhead |
| **Quantization** | 4-bit NormalFloat (NF4) | Loads base model in ~1.2 GB VRAM |
| **Fine-Tuning Framework** | Hugging Face `peft` | Low-Rank Adaptation (LoRA) |
| **LoRA Rank ($r$)** | 16 | Sufficient rank capacity for domain terminology |
| **LoRA Alpha ($\alpha$)** | 32 | Scaling factor ($\alpha / r = 2.0$) |
| **Target Modules** | `q_proj`, `k_proj`, `v_proj`, `o_proj` | Adapts query, key, value, and output attention heads |
| **Trainable Parameters** | < 1.5% of total parameters | Prevents catastrophic forgetting of base capabilities |
| **Compute Precision** | `fp16` | Optimized for NVIDIA T4 Tensor Cores |
| **Optimizer** | `AdamW` (PyTorch native) | Learning rate: 3e-4 with Cosine decay |

---

## 4. Model Highlights & Deliverables
* **Structured Legal Output:** Converts complex statutory text into structured bullet points with identified entity names, financial authorizations, and jurisdictional mandates.
* **Ultra-Lightweight Deployment:** Only the LoRA adapter matrix (`adapter_model.safetensors`, ~35–45 MB) needs to be stored and transferred, rather than re-distributing a multi-gigabyte foundation model.
* **Modular Hot-Swapping:** Demonstrates dynamic loading via `PeftModel.from_pretrained()`, allowing the legal adapter to be attached or detached from the base model on the fly.

In [1]:
# Clean install of required NLP & PEFT libraries
!pip uninstall -y torchvision torchaudio
!pip install -q transformers peft datasets accelerate bitsandbytes

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.1 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

# Load FiscalNote/billsum benchmark
dataset = load_dataset("FiscalNote/billsum")


# Standard PEFT instruction format
def format_legal_data(sample):
  formatted = []
  for text, summary in zip(sample["text"], sample["summary"]):
    prompt = (
        f"<|im_start|>system\nYou are a legal expert. Summarize the following"
        f" legislative bill into clear, plain English.<|im_end|>\n"
        f"<|im_start|>user\n{text[:1000].strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n{summary.strip()}<|im_end|>"
    )
    formatted.append(prompt)
  return {"text": formatted}


# Select 500 samples for efficient fine-tuning
train_data = (
    dataset["train"].select(range(500)).map(format_legal_data, batched=True)
)
print("Dataset prepared. Sample prompt length:", len(train_data[0]["text"]))

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset prepared. Sample prompt length: 2742


In [3]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. 4-bit Quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. Load Tokenizer & Base Model
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)

# 3. Define PEFT LoRA Configuration
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

# 4. Wrap with PEFT
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [4]:
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)


# Tokenize
def tokenize_inputs(examples):
  return tokenizer(examples["text"], truncation=True, max_length=512)


tokenized_train = train_data.map(
    tokenize_inputs, batched=True, remove_columns=train_data.column_names
)

# Training Hyperparameters
training_args = TrainingArguments(
    output_dir="./peft-legal-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    optim="adamw_torch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_train,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer, mlm=False, pad_to_multiple_of=8
    ),
)

print("Starting PEFT training...")
trainer.train()

# Save PEFT adapter weights directly via PEFT method
model.save_pretrained("./peft-legal-final")
tokenizer.save_pretrained("./peft-legal-final")
print("PEFT adapter saved successfully to './peft-legal-final'!")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Starting PEFT training...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.634005
20,1.419610
30,1.432053
40,1.472972
50,1.403411
60,1.436423


PEFT adapter saved successfully to './peft-legal-final'!


In [7]:
from peft import PeftModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Reload base model in evaluation mode
eval_base = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# 2. Attach your saved PEFT adapter
peft_model = PeftModel.from_pretrained(eval_base, "./peft-legal-final")
peft_model.eval()

# 3. Test on Unseen Sample
test_text = dataset["test"][0]["text"][:1000]
eval_prompt = (
    f"<|im_start|>system\nYou are a legal expert. Summarize the following"
    f" legislative bill into clear, plain English.<|im_end|>\n"
    f"<|im_start|>user\n{test_text.strip()}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
  output_tokens = peft_model.generate(
      **inputs,
      max_new_tokens=150,
      temperature=0.3,
      top_p=0.9,
      repetition_penalty=1.1,
      pad_token_id=tokenizer.eos_token_id,
      eos_token_id=tokenizer.eos_token_id,
      do_sample=True,
  )

response = tokenizer.decode(
    output_tokens[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
)

print("=" * 60)
print("🤖 PEFT FINE-TUNED MODEL SUMMARY:")
print("=" * 60)
print(response.strip())

print("\n" + "=" * 60)
print("📄 GROUND TRUTH SUMMARY:")
print("=" * 60)
print(dataset["test"][0]["summary"][:500] + "...")
print("=" * 60)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🤖 PEFT FINE-TUNED MODEL SUMMARY:
Environmental Infrastructure:  Amends the Water Resources Development Act of 1992 to provide $20 million for each of the following projects:

  * For the City of Jackson, Mississippi, to construct an alternative water supply system and a project for the elimination or control of combined sewer overflows;
  * For the Town of Manchester, New Hampshire, to construct an alternative water supply system and a project for the elimination or control of combined sewer overflows;
  * For the City of Atlanta, Georgia, to construct an alternative water supply system and a project for the elimination or control of combined sewer overflows; and
  * For the cities of Paterson, Passaic County, and Newark, New Jersey, to construct an alternative water

📄 GROUND TRUTH SUMMARY:
Amends the Water Resources Development Act of 1999 to: (1) authorize appropriations for FY 1999 through 2009 for implementation of a long-term resource monitoring program with respect to the Upper 